#### **IMPORT DE BIBLIOTECAS**

In [36]:
import pandas as pd 
import numpy as np

import seaborn as sns 
import matplotlib.pyplot as plt 

import warnings
warnings.filterwarnings('ignore')


import sweetviz as sv



pd.set_option('display.max_columns', None)

sns.set()


#### **SEPARANDO DADOS EM TREINO, VALIDACAO E TESTE**

In [37]:
df = pd.read_csv('../data/00_bronze/dataset_trat.csv', sep=';')

In [38]:
# CRINADO DATASET DE TRINAMENTO
treino = (
    df 
        .query('SAFRA < 200604')
)

# CRIANDO DATASET DE VALIDACAO
validacao = (
    df 
        .query('SAFRA in [200604, 200605]')
)

# CRIANDO DATASET DE TESTE
teste = (
    df 
        .query('SAFRA in [200606, 200607]')
)


treino.to_parquet('../data/00_bronze/treino.parquet')
validacao.to_parquet('../data/00_bronze/validacao.parquet')
teste.to_parquet('../data/00_bronze/teste.parquet')


##### **RELATÓRIO COM VISÃO GERAL DOS DADOSS**

In [39]:
report = sv.analyze(treino, target_feat='default')
report.show_html('../reports/sweetviz_treino.html')

                                             |          | [  0%]   00:00 -> (? left)

findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfo

Report ../reports/sweetviz_treino.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


#### **TRATAMENTO INICIAL DAS VARIAVEIS**

BILL_AMT - Foram encontrados valores menores do que 0. A estratégia aqui é substituir estes valores por -1.

PAY: Foram encontrados valores muito discrepantes. A estratégia aqui é substituir estes valores pelo percentil 90.

PAY_AMT: Foram encontrados valores muito discrepantes. A estratégia aqui é substituir estes valores extremos pelo percentil 90.

Os limites são calculados **apenas no treino** e aplicados em treino, validação e teste.

In [42]:
cols_pay = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
cols_pay_amt = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

limites = {
    col: treino[col].quantile(0.90)
    for col in cols_pay + cols_pay_amt
}


In [43]:
def aplicar_tratamento(base, limites):
    return base.assign(
        BILL_AMT1=lambda x: np.where(x['BILL_AMT1'] < 0, -1, x['BILL_AMT1']),
        BILL_AMT2=lambda x: np.where(x['BILL_AMT2'] < 0, -1, x['BILL_AMT2']),
        BILL_AMT3=lambda x: np.where(x['BILL_AMT3'] < 0, -1, x['BILL_AMT3']),
        BILL_AMT4=lambda x: np.where(x['BILL_AMT4'] < 0, -1, x['BILL_AMT4']),
        BILL_AMT5=lambda x: np.where(x['BILL_AMT5'] < 0, -1, x['BILL_AMT5']),
        BILL_AMT6=lambda x: np.where(x['BILL_AMT6'] < 0, -1, x['BILL_AMT6']),

        PAY_0=lambda x: np.where(x['PAY_0'] > limites['PAY_0'], limites['PAY_0'], x['PAY_0']),
        PAY_2=lambda x: np.where(x['PAY_2'] > limites['PAY_2'], limites['PAY_2'], x['PAY_2']),
        PAY_3=lambda x: np.where(x['PAY_3'] > limites['PAY_3'], limites['PAY_3'], x['PAY_3']),
        PAY_4=lambda x: np.where(x['PAY_4'] > limites['PAY_4'], limites['PAY_4'], x['PAY_4']),
        PAY_5=lambda x: np.where(x['PAY_5'] > limites['PAY_5'], limites['PAY_5'], x['PAY_5']),
        PAY_6=lambda x: np.where(x['PAY_6'] > limites['PAY_6'], limites['PAY_6'], x['PAY_6']),

        PAY_AMT1=lambda x: np.where(x['PAY_AMT1'] > limites['PAY_AMT1'], limites['PAY_AMT1'], x['PAY_AMT1']),
        PAY_AMT2=lambda x: np.where(x['PAY_AMT2'] > limites['PAY_AMT2'], limites['PAY_AMT2'], x['PAY_AMT2']),
        PAY_AMT3=lambda x: np.where(x['PAY_AMT3'] > limites['PAY_AMT3'], limites['PAY_AMT3'], x['PAY_AMT3']),
        PAY_AMT4=lambda x: np.where(x['PAY_AMT4'] > limites['PAY_AMT4'], limites['PAY_AMT4'], x['PAY_AMT4']),
        PAY_AMT5=lambda x: np.where(x['PAY_AMT5'] > limites['PAY_AMT5'], limites['PAY_AMT5'], x['PAY_AMT5']),
        PAY_AMT6=lambda x: np.where(x['PAY_AMT6'] > limites['PAY_AMT6'], limites['PAY_AMT6'], x['PAY_AMT6']),
    )

treino_tratado = aplicar_tratamento(treino, limites)
validacao_tratado = aplicar_tratamento(validacao, limites)
teste_tratado = aplicar_tratamento(teste, limites)

treino_tratado.to_parquet('../data/01_prata/treino_tratado.parquet')
validacao_tratado.to_parquet('../data/01_prata/validacao_tratado.parquet')
teste_tratado.to_parquet('../data/01_prata/teste_tratado.parquet')

pd.Series(limites).to_json('../data/01_prata/limites_tratamento.json')

print('treino', treino_tratado.shape, round(treino_tratado['default'].mean(), 4))
print('validacao', validacao_tratado.shape, round(validacao_tratado['default'].mean(), 4))
print('teste', teste_tratado.shape, round(teste_tratado['default'].mean(), 4))


treino (18000, 26) 0.2306
validacao (6000, 26) 0.2033
teste (6000, 26) 0.211
